# 09 — Consensus GRAD-CAM & EAA-IoU

Computes GRAD-CAM heatmaps for CNN, InceptionV3, and Xception on all test images,
then calculates the **Ensemble Attention Agreement IoU (EAA-IoU)** — a novel inter-model
saliency consensus metric introduced in this research.

Run notebooks 01, 03, 05 first to produce the model `.h5` files.

## Section 0 — Colab / Local Setup

Detects environment, installs packages, and configures Kaggle + HuggingFace credentials.

**Google Colab Secrets required** (Colab → 🔑 Secrets panel):

| Secret name | Value |
|---|---|
| `KAGGLE_USERNAME` | `sk1285` |
| `KAGGLE_KEY` | `7261c6b4046a6bd5c9ba4d1a6f58c98f` |
| `HF_TOKEN` | *(your HuggingFace write token)* |


In [ ]:
import sys, os, json, subprocess

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    subprocess.run(["pip", "install", "kaggle", "huggingface_hub", "-q"], check=False)
    from google.colab import userdata
    _kaggle_user = userdata.get('KAGGLE_USERNAME')
    _kaggle_key  = userdata.get('KAGGLE_KEY')
    HF_TOKEN     = userdata.get('HF_TOKEN')
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as _f:
        json.dump({'username': _kaggle_user, 'key': _kaggle_key}, _f)
    os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
    DATASET_PATH     = "/content/MRI_DATASET/"
    SAVED_MODELS_DIR = "/content/saved_models/"
    RESULTS_DIR      = "/content/results/"
    if not os.path.exists(os.path.join(DATASET_PATH, "Testing")):
        subprocess.run(["kaggle", "datasets", "download",
                        "masoudnickparvar/brain-tumor-mri-dataset",
                        "-p", "/content/"], check=False)
        subprocess.run(["unzip", "-q", "/content/brain-tumor-mri-dataset.zip",
                        "-d", DATASET_PATH], check=False)
        if os.path.exists("/content/brain-tumor-mri-dataset.zip"):
            os.remove("/content/brain-tumor-mri-dataset.zip")
else:
    DATASET_PATH     = "../MRI_DATASET/"
    SAVED_MODELS_DIR = "../saved_models/"
    RESULTS_DIR      = "../results/"
    HF_TOKEN         = os.environ.get('HF_TOKEN', '')

os.makedirs(SAVED_MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print("Environment :", "Colab" if IN_COLAB else "Local")
print("Dataset     :", DATASET_PATH)
print("Models      :", SAVED_MODELS_DIR)


## Section 1 — Imports

In [ ]:
import os, sys, numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import cv2
import tensorflow as tf
from tensorflow import keras
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

tf.random.set_seed(42)
np.random.seed(42)
print("TF version:", tf.__version__)


## Section 2 — Constants

In [ ]:
NOTEBOOK_NAME    = "09_ConsensusGradCAM"
HF_REPO_ID       = "shehank98/brain-tumor-mri-models"
CLASS_NAMES      = ["glioma", "meningioma", "notumor", "pituitary"]
IMG_SIZE_CNN     = (224, 224)
IMG_SIZE_PRETRAINED = (299, 299)
HEATMAP_SIZE     = (112, 112)   # common size for pairwise IoU
IOU_THRESHOLD    = 0.5          # binarisation threshold for heatmaps
RANDOM_SEED      = 42

DATASET_PATH     = globals().get("DATASET_PATH",     "../MRI_DATASET/")
SAVED_MODELS_DIR = globals().get("SAVED_MODELS_DIR", "../saved_models/")
RESULTS_DIR      = globals().get("RESULTS_DIR",      "../results/")
HF_TOKEN         = globals().get("HF_TOKEN",         os.environ.get("HF_TOKEN", ""))

RESULTS_NB_DIR   = os.path.join(RESULTS_DIR, NOTEBOOK_NAME)
os.makedirs(RESULTS_NB_DIR, exist_ok=True)
print("Results dir:", RESULTS_NB_DIR)


## Section 3 — Download & Load DL Models

Downloads `cnn_model.h5`, `inceptionv3_model.h5`, `xception_model.h5` from HuggingFace
if they are not already present in `SAVED_MODELS_DIR`.

In [ ]:
def _download_model_from_hf(filename, repo_id, token):
    """Download a model file from HuggingFace Hub if not already present."""
    dest = os.path.join(SAVED_MODELS_DIR, filename)
    if os.path.exists(dest):
        print(f"  Found locally: {filename}")
        return dest
    try:
        from huggingface_hub import hf_hub_download, login as hf_login
        if token:
            hf_login(token=token, add_to_git_credential=False)
        path = hf_hub_download(repo_id=repo_id, filename=f"models/{filename}",
                               local_dir=SAVED_MODELS_DIR)
        print(f"  Downloaded: {filename}")
        return path
    except Exception as e:
        print(f"  Could not download {filename}: {e}")
        return None

print("Checking / downloading DL models …")
for _fname in ["cnn_model.h5", "inceptionv3_model.h5", "xception_model.h5"]:
    _download_model_from_hf(_fname, HF_REPO_ID, HF_TOKEN)


In [ ]:
def _load(name):
    p = os.path.join(SAVED_MODELS_DIR, name)
    if not os.path.exists(p):
        raise FileNotFoundError(f"Model not found: {p}\nRun notebooks 01, 03, 05 first (or download from HF).")
    m = keras.models.load_model(p)
    m.trainable = False
    return m

print("Loading CNN …")
cnn_model = _load("cnn_model.h5")
print("Loading InceptionV3 …")
inc_model = _load("inceptionv3_model.h5")
print("Loading Xception …")
xcp_model = _load("xception_model.h5")
print("All 3 DL models loaded.")


## Section 4 — GRAD-CAM Helper Functions

Uses `tf.GradientTape` on the last convolutional layer of each model.
EAA-IoU = mean pairwise IoU of the three binarised heatmaps.

In [ ]:
def _last_conv_name(model):
    """Return the name of the last Conv layer with 4-D output."""
    for layer in reversed(model.layers):
        try:
            shape = layer.output_shape
            if isinstance(shape, list):
                shape = shape[0]
            if len(shape) == 4:
                return layer.name
        except Exception:
            continue
    raise ValueError("No 4-D conv layer found in model.")

# Cache layer names once
_LAST_CONV = {
    "cnn": _last_conv_name(cnn_model),
    "inc": _last_conv_name(inc_model),
    "xcp": _last_conv_name(xcp_model),
}
print("Last conv layers:", _LAST_CONV)

def compute_gradcam(model, model_key, img_array):
    """
    Compute GRAD-CAM heatmap for a preprocessed image.
    img_array : np.float32 array, shape (1, H, W, 3), values in [0, 1]
    Returns   : (heatmap np.float32 (H,W) in [0,1], predicted_class_idx int)
    """
    conv_name = _LAST_CONV[model_key]
    grad_model = keras.models.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(conv_name).output, model.output]
    )
    img_tf = tf.cast(img_array, tf.float32)
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_tf, training=False)
        pred_idx = int(tf.argmax(preds[0]))
        class_score = preds[:, pred_idx]
    grads = tape.gradient(class_score, conv_out)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap = (conv_out[0] @ pooled[..., tf.newaxis]).numpy().squeeze()
    heatmap = np.maximum(heatmap, 0)
    mx = heatmap.max()
    if mx > 0:
        heatmap = heatmap / mx
    return heatmap.astype(np.float32), pred_idx

def binary_iou(a, b, t=IOU_THRESHOLD):
    """IoU between two heatmaps binarised at threshold t."""
    a_bin = (a >= t)
    b_bin = (b >= t)
    inter = np.logical_and(a_bin, b_bin).sum()
    union = np.logical_or(a_bin,  b_bin).sum()
    return float(inter / union) if union > 0 else 1.0

def eaa_iou(hm_a, hm_b, hm_c, t=IOU_THRESHOLD):
    """Mean pairwise IoU across 3 heatmaps (EAA-IoU)."""
    iou_ab = binary_iou(hm_a, hm_b, t)
    iou_ac = binary_iou(hm_a, hm_c, t)
    iou_bc = binary_iou(hm_b, hm_c, t)
    return float(np.mean([iou_ab, iou_ac, iou_bc])), iou_ab, iou_ac, iou_bc

def consensus_heatmap(hm_a, hm_b, hm_c, weights=(1, 1, 1)):
    """Weighted average of 3 heatmaps, normalised to [0, 1]."""
    w = np.array(weights, dtype=np.float32)
    w = w / w.sum()
    fused = w[0]*hm_a + w[1]*hm_b + w[2]*hm_c
    mx = fused.max()
    return fused / mx if mx > 0 else fused

def overlay(img_rgb, heatmap, alpha=0.45):
    """Overlay a heatmap on an RGB image (uint8). Returns RGB uint8."""
    h, w = img_rgb.shape[:2]
    hm_up = cv2.resize(heatmap, (w, h))
    hm_u8 = np.uint8(255 * hm_up)
    col    = cv2.applyColorMap(hm_u8, cv2.COLORMAP_JET)
    col_rgb = cv2.cvtColor(col, cv2.COLOR_BGR2RGB)
    img_u8 = img_rgb if img_rgb.dtype == np.uint8 else np.uint8(img_rgb * 255)
    return cv2.addWeighted(img_u8, 1 - alpha, col_rgb, alpha, 0)


## Section 5 — Compute GRAD-CAM for All Test Images

Iterates through `Testing/` folder. For each image:
1. Preprocess for each model's input size
2. Compute GRAD-CAM (uses predicted class — models' own focus region)
3. Resize heatmaps to 112×112 common size
4. Compute pairwise IoU and EAA-IoU

> **Runtime:** ~20–40 min on Colab GPU for all 2,063 test images.

In [ ]:
TEST_DIR = os.path.join(DATASET_PATH, "Testing")

records = []
sample_store = {cls: [] for cls in CLASS_NAMES}  # store up to 2 samples per class for viz

for cls_name in CLASS_NAMES:
    cls_dir = os.path.join(TEST_DIR, cls_name)
    if not os.path.isdir(cls_dir):
        print(f"  Warning: {cls_dir} not found"); continue
    img_files = sorted([f for f in os.listdir(cls_dir)
                        if f.lower().endswith(('.jpg','.jpeg','.png'))])
    for fname in tqdm(img_files, desc=cls_name):
        img_path = os.path.join(cls_dir, fname)
        img_bgr  = cv2.imread(img_path)
        if img_bgr is None:
            continue
        img_rgb  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

        # Preprocess for each architecture
        img_cnn = cv2.resize(img_rgb, IMG_SIZE_CNN).astype(np.float32) / 255.0
        img_299 = cv2.resize(img_rgb, IMG_SIZE_PRETRAINED).astype(np.float32) / 255.0
        arr_cnn = img_cnn[np.newaxis]
        arr_299 = img_299[np.newaxis]

        # GRAD-CAM
        hm_cnn, pred_cnn = compute_gradcam(cnn_model, "cnn", arr_cnn)
        hm_inc, pred_inc = compute_gradcam(inc_model, "inc", arr_299)
        hm_xcp, pred_xcp = compute_gradcam(xcp_model, "xcp", arr_299)

        # Resize to common size for IoU
        hm_cnn_r = cv2.resize(hm_cnn, HEATMAP_SIZE)
        hm_inc_r = cv2.resize(hm_inc, HEATMAP_SIZE)
        hm_xcp_r = cv2.resize(hm_xcp, HEATMAP_SIZE)

        eaa, iou_ci, iou_cx, iou_ix = eaa_iou(hm_cnn_r, hm_inc_r, hm_xcp_r)

        # Ensemble majority vote
        preds_list = [pred_cnn, pred_inc, pred_xcp]
        from collections import Counter
        majority_pred = Counter(preds_list).most_common(1)[0][0]

        rec = {
            "image_path": img_path,
            "true_class": cls_name,
            "true_idx":   CLASS_NAMES.index(cls_name),
            "pred_cnn":   CLASS_NAMES[pred_cnn],
            "pred_inc":   CLASS_NAMES[pred_inc],
            "pred_xcp":   CLASS_NAMES[pred_xcp],
            "majority_pred": CLASS_NAMES[majority_pred],
            "majority_correct": int(majority_pred == CLASS_NAMES.index(cls_name)),
            "eaa_iou":    round(eaa, 4),
            "iou_cnn_inc": round(iou_ci, 4),
            "iou_cnn_xcp": round(iou_cx, 4),
            "iou_inc_xcp": round(iou_ix, 4),
        }
        records.append(rec)

        # Store sample for viz (up to 3 per class)
        if len(sample_store[cls_name]) < 3:
            sample_store[cls_name].append({
                "img_rgb": img_rgb,
                "hm_cnn": hm_cnn, "hm_inc": hm_inc, "hm_xcp": hm_xcp,
                "consensus": consensus_heatmap(hm_cnn_r, hm_inc_r, hm_xcp_r),
                "eaa_iou": eaa,
                "majority_pred": CLASS_NAMES[majority_pred],
                "correct": majority_pred == CLASS_NAMES.index(cls_name),
            })

df = pd.DataFrame(records)
print(f"\nProcessed {len(df)} images.")
print(df[["true_class","pred_cnn","pred_inc","pred_xcp","eaa_iou"]].head(8))


## Section 6 — Save Results CSV

In [ ]:
csv_path = os.path.join(RESULTS_NB_DIR, "eaa_iou_results.csv")
df.to_csv(csv_path, index=False)
print(f"Saved CSV: {csv_path}")
print("EAA-IoU summary:")
print(df.groupby("true_class")["eaa_iou"].describe().round(3))


## Section 7 — Visualisation: Sample Heatmap Grid

In [ ]:
# --- Chart 1: Sample GRAD-CAM grid (4 classes × 5 columns: orig, CNN, Inc, Xcp, Consensus) ---
fig, axes = plt.subplots(4, 5, figsize=(18, 14))
col_titles = ["Original", "CNN GRAD-CAM", "InceptionV3 GRAD-CAM",
              "Xception GRAD-CAM", "Consensus Map"]
for col, title in enumerate(col_titles):
    axes[0, col].set_title(title, fontsize=11, fontweight='bold', pad=8)

for row_idx, cls_name in enumerate(CLASS_NAMES):
    samples = sample_store[cls_name]
    if not samples:
        continue
    s = samples[0]
    img_disp = cv2.resize(s["img_rgb"], (224, 224))
    axes[row_idx, 0].imshow(img_disp); axes[row_idx, 0].set_ylabel(cls_name.capitalize(), fontsize=11, fontweight='bold')
    axes[row_idx, 1].imshow(overlay(img_disp, cv2.resize(s["hm_cnn"], (224,224))))
    axes[row_idx, 2].imshow(overlay(img_disp, cv2.resize(s["hm_inc"], (224,224))))
    axes[row_idx, 3].imshow(overlay(img_disp, cv2.resize(s["hm_xcp"], (224,224))))
    con_up = cv2.resize(s["consensus"], (224,224))
    axes[row_idx, 4].imshow(overlay(img_disp, con_up))
    for ax in axes[row_idx]:
        ax.axis('off')
    axes[row_idx, 4].set_xlabel(f"EAA-IoU: {s['eaa_iou']:.3f}", fontsize=9)

fig.suptitle("GRAD-CAM Heatmaps — CNN · InceptionV3 · Xception · Consensus\n"
             "Brain Tumor MRI Dataset (Kaggle)", fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
p = os.path.join(RESULTS_NB_DIR, "sample_gradcam_grid.jpg")
plt.savefig(p, dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print("Saved:", p)


## Section 8 — Visualisation: EAA-IoU Distribution by Class

In [ ]:
# --- Chart 2: EAA-IoU distribution histogram by class ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = {'glioma':'#e74c3c','meningioma':'#e67e22','notumor':'#2ecc71','pituitary':'#3498db'}

ax = axes[0]
for cls in CLASS_NAMES:
    sub = df[df["true_class"] == cls]["eaa_iou"]
    ax.hist(sub, bins=30, alpha=0.6, label=cls.capitalize(), color=colors[cls], edgecolor='white')
ax.axvline(0.40, color='red',    linestyle='--', lw=1.5, label='Low/Mod threshold (0.40)')
ax.axvline(0.65, color='orange', linestyle='--', lw=1.5, label='Mod/High threshold (0.65)')
ax.set_xlabel("EAA-IoU", fontsize=11); ax.set_ylabel("Count", fontsize=11)
ax.set_title("EAA-IoU Distribution by Tumor Class", fontsize=12, fontweight='bold')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

ax = axes[1]
means = df.groupby("true_class")["eaa_iou"].mean().reindex(CLASS_NAMES)
stds  = df.groupby("true_class")["eaa_iou"].std().reindex(CLASS_NAMES)
bars  = ax.bar([c.capitalize() for c in CLASS_NAMES], means,
               yerr=stds, capsize=5, color=[colors[c] for c in CLASS_NAMES],
               alpha=0.8, edgecolor='white')
ax.set_ylim(0, 1); ax.set_ylabel("Mean EAA-IoU", fontsize=11)
ax.set_title("Mean EAA-IoU ± Std per Class", fontsize=12, fontweight='bold')
ax.axhline(0.65, color='orange', linestyle='--', lw=1.2, label='High-confidence threshold')
ax.axhline(0.40, color='red',    linestyle='--', lw=1.2, label='Escalation threshold')
ax.legend(fontsize=9); ax.grid(axis='y', alpha=0.3)
for bar, val in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{val:.3f}", ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
p = os.path.join(RESULTS_NB_DIR, "eaa_iou_distribution.jpg")
plt.savefig(p, dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print("Saved:", p)


## Section 9 — Visualisation: Low vs High EAA-IoU Examples

In [ ]:
# --- Chart 3: Low vs High EAA-IoU examples ---
df_sorted = df.sort_values("eaa_iou")
low_examples  = df_sorted.head(6)
high_examples = df_sorted.tail(6)

def load_and_show(row, ax, size=224):
    img = cv2.imread(row["image_path"])
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (size, size))
    ax.imshow(img); ax.axis('off')
    cls = row['true_class'].capitalize()
    iou = row['eaa_iou']
    correct = row['true_class'] == row['majority_pred']
    color = 'green' if correct else 'red'
    ax.set_title(f"{cls}\nIoU:{iou:.3f}", fontsize=8, color=color)

fig, axes = plt.subplots(2, 6, figsize=(18, 7))
for i, (_, row) in enumerate(low_examples.iterrows()):
    load_and_show(row, axes[0, i])
axes[0, 0].set_ylabel("Low EAA-IoU\n(Model Disagree)", fontsize=10, fontweight='bold', labelpad=10)

for i, (_, row) in enumerate(high_examples.iterrows()):
    load_and_show(row, axes[1, i])
axes[1, 0].set_ylabel("High EAA-IoU\n(Model Agree)", fontsize=10, fontweight='bold', labelpad=10)

fig.suptitle("Low vs High EAA-IoU Examples\n(Green title = correct prediction, Red = misclassified)",
             fontsize=12, fontweight='bold')
plt.tight_layout()
p = os.path.join(RESULTS_NB_DIR, "eaa_iou_low_vs_high.jpg")
plt.savefig(p, dpi=150, bbox_inches='tight')
plt.show(); plt.close()
print("Saved:", p)


## Section 10 — Upload to HuggingFace

In [ ]:
def _hf_upload(files, repo_id, token, prefix=""):
    from huggingface_hub import HfApi, login as hf_login
    if not token:
        print("No HF_TOKEN — skipping upload.")
        return
    hf_login(token=token, add_to_git_credential=False)
    api = HfApi()
    api.create_repo(repo_id=repo_id, repo_type="model", private=False, exist_ok=True)
    for p in files:
        if not os.path.exists(p):
            print(f"  skip (missing): {p}"); continue
        rp = (prefix + "/" + os.path.basename(p)).lstrip("/")
        api.upload_file(path_or_fileobj=p, path_in_repo=rp,
                        repo_id=repo_id, repo_type="model")
        print(f"  uploaded: {rp}")

files_to_upload = [
    os.path.join(RESULTS_NB_DIR, "eaa_iou_results.csv"),
    os.path.join(RESULTS_NB_DIR, "sample_gradcam_grid.jpg"),
    os.path.join(RESULTS_NB_DIR, "eaa_iou_distribution.jpg"),
    os.path.join(RESULTS_NB_DIR, "eaa_iou_low_vs_high.jpg"),
]
_hf_upload(files_to_upload, HF_REPO_ID, HF_TOKEN, prefix=f"results/{NOTEBOOK_NAME}")
print("\nSection 10 complete.")
